In [29]:
# feature list 

import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier

X_train = pd.read_csv('../data/processed/X_train.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()

# recreate RF importance ranking
rf_check = RandomForestClassifier(
    n_estimators=100, max_depth=4,
    class_weight='balanced', random_state=42
)
rf_check.fit(X_train, y_train)

fi = pd.Series(rf_check.feature_importances_, index=X_train.columns)
fi_top15 = fi.sort_values(ascending=False).head(15).index.tolist()

print("Top 15 features:")
print(fi_top15)

joblib.dump(fi_top15, '../models/best_model_features.pkl')
print("\nbest_model_features.pkl saved")

Top 15 features:
['health_score', 'total_usage_count', 'error_rate', 'feature_adoption_score', 'is_trial', 'mean_satisfaction', 'distinct_features', 'sub_count', 'total_errors', 'tenure_days', 'max_mrr', 'engagement_segment', 'mean_resolution_hrs', 'risk_score', 'beta_feature_usage']

best_model_features.pkl saved


In [39]:
# STRICT DASHBOARD EXPORT
import os
import joblib
import numpy as np
import pandas as pd

os.makedirs('../data/predictions', exist_ok=True)

master_df = pd.read_csv('../data/processed/master_df.csv')
df_preprocessed = pd.read_csv('../data/processed/df_preprocessed.csv')
subscriptions = pd.read_csv('../data/raw/subscriptions.csv')
best_model = joblib.load('../models/best_model.pkl')

if hasattr(best_model, 'feature_names_in_'):
    model_features = list(best_model.feature_names_in_)
else:
    model_features = pd.read_csv('../data/processed/X_train.csv').columns.tolist()

X_all = df_preprocessed.copy()
for col in model_features:
    if col not in X_all.columns:
        X_all[col] = 0
X_model = X_all[model_features].apply(pd.to_numeric, errors='coerce').fillna(0)

churn_probability = best_model.predict_proba(X_model)[:, 1]
predicted_churn = (churn_probability >= 0.5).astype(int)

dashboard = pd.DataFrame()
dashboard['account_id'] = master_df['account_id'].astype('string')
dashboard['account_name'] = master_df['account_name'].astype('string')
dashboard['industry'] = master_df['industry'].astype('string')
dashboard['country'] = master_df['country'].astype('string')
dashboard['plan_tier'] = master_df['plan_tier'].astype('string')

signup_dt = pd.to_datetime(master_df['signup_date'], errors='coerce')
dashboard['signup_date'] = signup_dt.fillna(pd.Timestamp('2024-01-01')).dt.strftime('%Y-%m-%d')

dashboard['tenure_days'] = pd.to_numeric(df_preprocessed['tenure_days'], errors='coerce').fillna(0)
dashboard['max_mrr'] = pd.to_numeric(master_df['max_mrr'], errors='coerce').fillna(0)
dashboard['is_trial'] = pd.to_numeric(master_df['is_trial'], errors='coerce').fillna(0).astype(int).clip(0, 1)

billing = master_df['billing_frequency'].astype('string').str.lower()
billing_map = {'0': 'annual', '1': 'monthly'}
dashboard['billing_frequency'] = billing.map(lambda x: billing_map.get(x, x))
dashboard['billing_frequency'] = dashboard['billing_frequency'].replace({'<NA>': 'monthly'}).fillna('monthly')

dashboard['total_usage_count'] = pd.to_numeric(df_preprocessed['total_usage_count'], errors='coerce').fillna(0)
dashboard['mean_satisfaction'] = pd.to_numeric(df_preprocessed['mean_satisfaction'], errors='coerce').fillna(3.0).clip(1.0, 5.0)
dashboard['health_score'] = pd.to_numeric(df_preprocessed['health_score'], errors='coerce').fillna(0).clip(0, 100)
dashboard['risk_score'] = pd.to_numeric(df_preprocessed['risk_score'], errors='coerce').fillna(0).clip(0, 100)

dashboard['engagement_segment'] = (
    df_preprocessed['engagement_segment']
    .astype('string')
    .replace({'<NA>': np.nan})
    .fillna('Healthy')
)

dashboard['actual_churn'] = pd.to_numeric(master_df['churn_flag'], errors='coerce').fillna(0).astype(int).clip(0, 1)
dashboard['churn_probability'] = pd.Series(churn_probability).clip(0, 1).round(4)
dashboard['predicted_churn'] = pd.Series(predicted_churn).astype(int).clip(0, 1)
dashboard['risk_category'] = pd.cut(
    dashboard['churn_probability'],
    bins=[-0.001, 0.35, 0.65, 1.0],
    labels=['Low', 'Medium', 'High'],
    include_lowest=True,
 ).astype('string')

# Revenue at risk = sum of all subscription MRRs under each account.
subscriptions['account_id'] = subscriptions['account_id'].astype('string')
subscriptions['mrr_amount'] = pd.to_numeric(subscriptions['mrr_amount'], errors='coerce').fillna(0)
account_revenue = subscriptions.groupby('account_id', as_index=False)['mrr_amount'].sum()
account_revenue = account_revenue.rename(columns={'mrr_amount': 'revenue_at_risk'})
dashboard = dashboard.merge(account_revenue, on='account_id', how='left')
dashboard['revenue_at_risk'] = pd.to_numeric(dashboard['revenue_at_risk'], errors='coerce').fillna(0)

dashboard_cols = [
    'account_id', 'account_name', 'industry', 'country',
    'plan_tier', 'signup_date', 'tenure_days', 'max_mrr',
    'is_trial', 'billing_frequency', 'total_usage_count',
    'mean_satisfaction', 'health_score', 'risk_score',
    'engagement_segment', 'actual_churn', 'churn_probability',
    'predicted_churn', 'risk_category', 'revenue_at_risk',
]
dashboard = dashboard[dashboard_cols]

dashboard = dashboard.astype(
    {
        'account_id': 'string',
        'account_name': 'string',
        'industry': 'string',
        'country': 'string',
        'plan_tier': 'string',
        'signup_date': 'string',
        'billing_frequency': 'string',
        'engagement_segment': 'string',
        'risk_category': 'string',
        'is_trial': 'int64',
        'actual_churn': 'int64',
        'predicted_churn': 'int64',
    },
    errors='ignore',
)

dashboard.to_csv('../data/predictions/dashboard.csv', index=False)

print('dashboard.csv saved - shape:', dashboard.shape)


print('Columns:', dashboard.columns.tolist())
print('\nDtypes:')
print(dashboard.dtypes)
print('\nFirst 5 rows (all columns):')
with pd.option_context('display.max_columns', None, 'display.width', 2000):
    print(dashboard.head(5).to_string(index=False))

dashboard.csv saved - shape: (500, 20)
Columns: ['account_id', 'account_name', 'industry', 'country', 'plan_tier', 'signup_date', 'tenure_days', 'max_mrr', 'is_trial', 'billing_frequency', 'total_usage_count', 'mean_satisfaction', 'health_score', 'risk_score', 'engagement_segment', 'actual_churn', 'churn_probability', 'predicted_churn', 'risk_category', 'revenue_at_risk']

Dtypes:
account_id             string
account_name           string
industry               string
country                string
plan_tier              string
signup_date            string
tenure_days             int64
max_mrr                 int64
is_trial                int64
billing_frequency      string
total_usage_count       int64
mean_satisfaction     float64
health_score          float64
risk_score            float64
engagement_segment     string
actual_churn            int64
churn_probability     float64
predicted_churn         int64
risk_category          string
revenue_at_risk         int64
dtype: object

F